In [ ]:
from pathlib import Path
import subprocess
import wave

import imageio_ffmpeg
import pandas as pd

root = Path("data") if Path("data/karakalpak-audio-dataset-hf/train.csv").exists() else Path("../data")
hf_dir = root / "karakalpak-audio-dataset-hf"
splits = ("train", "test")
ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()

In [ ]:
records = {split: pd.read_csv(hf_dir / f"{split}.csv") for split in splits}
for split, frame in records.items():
    assert set(frame["filename"]) == {p.name for p in (hf_dir / split).glob("*.wav")}
{split: len(frame) for split, frame in records.items()}

In [ ]:
for split in splits:
    source_dir = hf_dir / split
    output_dir = hf_dir / f"{split}_16k"
    output_dir.mkdir(exist_ok=True)
    for source in sorted(source_dir.glob("*.wav")):
        destination = output_dir / source.name
        temporary = destination.with_suffix(".wav.partial")
        subprocess.run(
            [ffmpeg, "-nostdin", "-loglevel", "error", "-threads", "1", "-y",
             "-i", str(source), "-ar", "16000", "-ac", "1",
             "-c:a", "pcm_s16le", "-f", "wav", str(temporary)],
            check=True,
        )
        temporary.replace(destination)

In [ ]:
for split, frame in records.items():
    source_dir = hf_dir / split
    output_dir = hf_dir / f"{split}_16k"
    assert {p.name for p in output_dir.glob("*.wav")} == set(frame["filename"])
    for name in frame["filename"]:
        with wave.open(str(source_dir / name), "rb") as original, wave.open(str(output_dir / name), "rb") as converted:
            assert original.getframerate() == 44_100
            assert (converted.getframerate(), converted.getnchannels(), converted.getsampwidth()) == (16_000, 1, 2)
            assert abs(original.getnframes() / 44_100 - converted.getnframes() / 16_000) < 1 / 16_000
{split: len(frame) for split, frame in records.items()}